# Comprension del Negocio y de Datos

## Exploración previa en Excel (tablas dinámicas)

Antes de tocar código, se armaron tres tablas dinámicas sobre el dataset crudo para
tener una intuición inicial de qué variables podrían ser predictivas:

- **Churn x Contrato**: relación fuertemente inversa entre compromiso contractual y fuga
  (Month-to-month: 42.7% churn vs. Two year: 2.8% churn — ~15x de diferencia).
- **Churn x Método de pago**: los métodos automáticos (Bank transfer, Credit card) muestran
  churn bajo (~15-17%), mientras que Electronic check se dispara a 45.7% — posible señal de
  fricción operativa como causa de fuga distinta al compromiso contractual.
- **Churn x Antigüedad**: relación monótona decreciente. El primer año de vida del cliente
  concentra el mayor riesgo (48.3% churn en 0-11 meses vs. 6.7% en 60-72 meses).

Estas tres variables (`Contract`, `PaymentMethod`, `tenure`) son las primeras candidatas
fuertes para el modelo, a confirmar con `feature_importances_` en la Etapa 4.

## Diccionario de datos

Resumen de las 33 columnas del dataset, con las decisiones sobre su uso en el proyecto.
El detalle de cómo se llegó a cada decisión está documentado en la sección de auditoría
más abajo.

| Columna | Tipo de dato | Descripción | Uso en el modelo |
|---|---|---|---|
| `CustomerID` | Texto (identificador) | ID único de cada cliente | Descartar — identificador, no predictor |
| `Count` | Numérico | Siempre vale 1 (auxiliar de IBM) | Descartar — constante |
| `Country` | Categórica | País (siempre "United States") | Descartar — constante |
| `State` | Categórica | Estado (siempre "California") | Descartar — constante |
| `City` | Categórica (alta cardinalidad) | Ciudad del cliente (1,129 valores únicos) | Descartar — cardinalidad muy alta, fuera de foco |
| `Zip Code` | Numérico | Código postal | Descartar — geográfica, fuera de foco |
| `Lat Long` | Texto | Latitud y longitud combinadas en un string | Descartar — geográfica, redundante con Latitude/Longitude |
| `Latitude` | Numérico | Latitud geográfica | Descartar — geográfica, fuera de foco |
| `Longitude` | Numérico | Longitud geográfica | Descartar — geográfica, fuera de foco |
| `Gender` | Categórica binaria | Male / Female | Predictora — encoding 0/1 |
| `Senior Citizen` | Categórica binaria | Si el cliente es adulto mayor | Predictora — encoding 0/1 |
| `Partner` | Categórica binaria | Si tiene pareja | Predictora — encoding 0/1 |
| `Dependents` | Categórica binaria | Si tiene personas a cargo | Predictora — encoding 0/1 |
| `Tenure Months` | Numérico | Meses de antigüedad como cliente | Predictora — numérica directa |
| `Phone Service` | Categórica binaria | Si tiene servicio telefónico | Predictora — encoding 0/1 |
| `Multiple Lines` | Categórica (3 valores) | Yes / No / No phone service | Predictora — one-hot |
| `Internet Service` | Categórica (3 valores) | DSL / Fiber optic / No | Predictora — one-hot |
| `Online Security` | Categórica (3 valores) | Yes / No / No internet service | Predictora — one-hot (revisar fusión de categorías) |
| `Online Backup` | Categórica (3 valores) | Yes / No / No internet service | Predictora — one-hot (revisar fusión de categorías) |
| `Device Protection` | Categórica (3 valores) | Yes / No / No internet service | Predictora — one-hot (revisar fusión de categorías) |
| `Tech Support` | Categórica (3 valores) | Yes / No / No internet service | Predictora — one-hot (revisar fusión de categorías) |
| `Streaming TV` | Categórica (3 valores) | Yes / No / No internet service | Predictora — one-hot (revisar fusión de categorías) |
| `Streaming Movies` | Categórica (3 valores) | Yes / No / No internet service | Predictora — one-hot (revisar fusión de categorías) |
| `Contract` | Categórica (3 valores, ordinal) | Month-to-month / One year / Two year | Predictora — codificación ordinal (evaluar vs. one-hot) |
| `Paperless Billing` | Categórica binaria | Si factura sin papel | Predictora — encoding 0/1 |
| `Payment Method` | Categórica (4 valores) | Forma de pago del cliente | Predictora — one-hot |
| `Monthly Charges` | Numérico | Cargo mensual actual | Predictora — numérica directa (evaluar escalado) |
| `Total Charges` | Numérico (mal tipado como texto) | Cargo total histórico acumulado | Predictora — convertir a numérico, imputar 11 casos con 0 |
| `Churn Label` | Categórica binaria | Yes / No — si el cliente se dio de baja | **Target** (versión texto) |
| `Churn Value` | Numérico binario | 1 / 0 — mismo target en formato numérico | **Target** (versión numérica, usar esta para el modelo) |
| `Churn Score` | Numérico | Score de propensión calculado por otro modelo (IBM) | Descartar — data leakage |
| `CLTV` | Numérico | Customer Lifetime Value calculado por IBM | Evaluar en Etapa 5 (impacto económico), no como predictora |
| `Churn Reason` | Categórica (20 valores) | Razón declarada de la baja | Descartar del modelo (no disponible para clientes activos) — usar en EDA/README |

## 1. Cargar y confirmar dimensiones

In [2]:
import pandas as pd

df=pd.read_excel('../data/raw/Telco_customer_churn.xlsx', sheet_name='Telco_Churn')
df.shape

(7043, 33)

## 2. Ver columnas y primeras filas

In [3]:
df.columns.to_list()

['CustomerID',
 'Count',
 'Country',
 'State',
 'City',
 'Zip Code',
 'Lat Long',
 'Latitude',
 'Longitude',
 'Gender',
 'Senior Citizen',
 'Partner',
 'Dependents',
 'Tenure Months',
 'Phone Service',
 'Multiple Lines',
 'Internet Service',
 'Online Security',
 'Online Backup',
 'Device Protection',
 'Tech Support',
 'Streaming TV',
 'Streaming Movies',
 'Contract',
 'Paperless Billing',
 'Payment Method',
 'Monthly Charges',
 'Total Charges',
 'Churn Label',
 'Churn Value',
 'Churn Score',
 'CLTV',
 'Churn Reason']

In [4]:
df[['Country', 'State', 'Count']].nunique()

Country    1
State      1
Count      1
dtype: int64

### Auditoría de columnas

El dataset trae 33 columnas (versión extendida de IBM), no las 21 "clásicas" del dataset
básico. Se identificaron 3 columnas constantes (`Country`, `State`, `Count`, confirmado con
`.nunique()`) sin valor predictivo, además de columnas geográficas (`City`, `Zip Code`,
`Lat Long`, `Latitude`, `Longitude`) fuera del foco de este análisis. Se descartan también
`Churn Score` (data leakage: predicción de otro modelo) y `Churn Reason` (no disponible para
clientes activos). El target será `Churn Value` (o `Churn Label`), no ambos simultáneamente.

In [5]:
df.head(5)

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


## 3. Tipos de datos

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         7043 non-null   str    
 1   Count              7043 non-null   int64  
 2   Country            7043 non-null   str    
 3   State              7043 non-null   str    
 4   City               7043 non-null   str    
 5   Zip Code           7043 non-null   int64  
 6   Lat Long           7043 non-null   str    
 7   Latitude           7043 non-null   float64
 8   Longitude          7043 non-null   float64
 9   Gender             7043 non-null   str    
 10  Senior Citizen     7043 non-null   str    
 11  Partner            7043 non-null   str    
 12  Dependents         7043 non-null   str    
 13  Tenure Months      7043 non-null   int64  
 14  Phone Service      7043 non-null   str    
 15  Multiple Lines     7043 non-null   str    
 16  Internet Service   7043 non-null   

In [7]:
df['Total Charges'].dtype

dtype('O')

In [8]:
df['Total Charges'].apply(lambda x: not str(x).strip().replace('.','').isdigit()).sum()

np.int64(11)

In [9]:
df[df['Total Charges'].apply(lambda x: not str(x).strip().replace('.','').isdigit())]

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
2234,4472-LVYGI,1,United States,California,San Bernardino,92408,"34.084909, -117.258107",34.084909,-117.258107,Female,...,Two year,Yes,Bank transfer (automatic),52.55,,No,0,36,2578,NaN
2438,3115-CZMZD,1,United States,California,Independence,93526,"36.869584, -118.189241",36.869584,-118.189241,Male,...,Two year,No,Mailed check,20.25,,No,0,68,5504,NaN
2568,5709-LVOEQ,1,United States,California,San Mateo,94401,"37.590421, -122.306467",37.590421,-122.306467,Female,...,Two year,No,Mailed check,80.85,,No,0,45,2048,NaN
2667,4367-NUYAO,1,United States,California,Cupertino,95014,"37.306612, -122.080621",37.306612,-122.080621,Male,...,Two year,No,Mailed check,25.75,,No,0,48,4950,NaN
2856,1371-DWPAZ,1,United States,California,Redcrest,95569,"40.363446, -123.835041",40.363446,-123.835041,Female,...,Two year,No,Credit card (automatic),56.05,,No,0,30,4740,NaN
4331,7644-OMVMY,1,United States,California,Los Angeles,90029,"34.089953, -118.294824",34.089953,-118.294824,Male,...,Two year,No,Mailed check,19.85,,No,0,53,2019,NaN
4687,3213-VVOLG,1,United States,California,Sun City,92585,"33.739412, -117.173334",33.739412,-117.173334,Male,...,Two year,No,Mailed check,25.35,,No,0,49,2299,NaN
5104,2520-SGTTA,1,United States,California,Ben Lomond,95005,"37.078873, -122.090386",37.078873,-122.090386,Female,...,Two year,No,Mailed check,20.00,,No,0,27,3763,NaN
5719,2923-ARZLG,1,United States,California,La Verne,91750,"34.144703, -117.770299",34.144703,-117.770299,Male,...,One year,Yes,Mailed check,19.70,,No,0,69,4890,NaN
6772,4075-WKNIU,1,United States,California,Bell,90201,"33.970343, -118.171368",33.970343,-118.171368,Female,...,Two year,No,Mailed check,73.35,,No,0,44,2342,NaN


In [10]:
df[df['Total Charges'].apply(lambda x: not str(x).strip().replace('.','').isdigit())]['Tenure Months']

2234    0
2438    0
2568    0
2667    0
2856    0
4331    0
4687    0
5104    0
5719    0
6772    0
6840    0
Name: Tenure Months, dtype: int64

### Auditoría de `Total Charges`

`Total Charges` está tipada como `object` (texto) en vez de numérica. Se identificaron 11
registros con valor en blanco, correspondientes en el 100% de los casos a clientes con
`Tenure Months = 0` (recién dados de alta, que todavía no generaron cargo total acumulado).
Todos estos casos tienen `Churn Label = No`.

**Decisión**: convertir `Total Charges` a numérica e imputar estos 11 casos con `0`,
consistente con la lógica de negocio (cliente nuevo sin historial de facturación).
Esta transformación se ejecuta en la Etapa 3 (Preparación de datos), no acá — esta etapa
es solo de diagnóstico.

## 4. Valores Nulos

In [11]:
df.isnull().sum()

CustomerID              0
Count                   0
Country                 0
State                   0
City                    0
Zip Code                0
Lat Long                0
Latitude                0
Longitude               0
Gender                  0
Senior Citizen          0
Partner                 0
Dependents              0
Tenure Months           0
Phone Service           0
Multiple Lines          0
Internet Service        0
Online Security         0
Online Backup           0
Device Protection       0
Tech Support            0
Streaming TV            0
Streaming Movies        0
Contract                0
Paperless Billing       0
Payment Method          0
Monthly Charges         0
Total Charges           0
Churn Label             0
Churn Value             0
Churn Score             0
CLTV                    0
Churn Reason         5174
dtype: int64

In [12]:
df['Total Charges'].isnull().sum()

np.int64(0)

### Nulos generales

`.isnull().sum()` no detecta problema alguno en `Total Charges` (0 nulos), a pesar de que
ya se confirmó que tiene 11 valores en blanco no numéricos. Esto se debe a que esos valores
son strings vacíos, no `NaN` reales — un nulo "silencioso" que solo se detecta revisando el
contenido real de la columna, no solo su método `.isnull()`.

El único nulo genuino del dataset es `Churn Reason` (5,174 nulos), que coincide exactamente
con la cantidad de clientes que no se fueron (7,043 − 1,869 = 5,174) — confirma que esta
columna solo existe para clientes con `Churn Label = Yes`.

## 5. Balance de clases

chequeo más importante de toda la etapa. Ya vimos en Excel que el churn general es ~26,5%/73,5% — necesitamos confirmarlo también en código porque esto va a determinar decisiones clave más adelante

In [13]:
df['Churn Label'].value_counts(normalize=True)

Churn Label
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64

### Balance de clases

El dataset tiene desbalance de clases moderado: 73.46% No Churn vs. 26.54% Yes Churn
(confirmado, coincide con la exploración previa en Excel). No es un desbalance extremo,
pero sí lo suficiente como para que el **accuracy no sea una métrica confiable** — un
modelo que prediga siempre "No" tendría ~73% de accuracy sin haber aprendido nada útil.

**Implicancias para etapas posteriores**:
- Etapa 4 (Modelado): usar **Stratified K-Fold** en vez de K-Fold simple, para que cada
  fold mantenga esta misma proporción.
- Evaluar `class_weight='balanced'` como primera opción antes de recurrir a SMOTE.
- Las métricas de referencia van a ser Recall, Precision y F1 de la clase Yes, no accuracy.

## 6. Cardinalidad de las categóricas

define qué tipo de encoding usar en la Etapa 3 (binario, one-hot, ordinal). Una columna con 50 valores únicos no se codifica igual que una con 2.

In [14]:
df.select_dtypes(exclude='number').nunique()

CustomerID           7043
Country                 1
State                   1
City                 1129
Lat Long             1652
Gender                  2
Senior Citizen          2
Partner                 2
Dependents              2
Phone Service           2
Multiple Lines          3
Internet Service        3
Online Security         3
Online Backup           3
Device Protection       3
Tech Support            3
Streaming TV            3
Streaming Movies        3
Contract                3
Paperless Billing       2
Payment Method          4
Total Charges        6531
Churn Label             2
Churn Reason           20
dtype: int64

### Cardinalidad de variables categóricas

- **Binarias (2 valores)**: Gender, Senior Citizen, Partner, Dependents, Phone Service,
  Paperless Billing → encoding 0/1 directo.
- **Baja cardinalidad (3 valores)**: Multiple Lines, Internet Service, Online Security,
  Online Backup, Device Protection, Tech Support, Streaming TV, Streaming Movies, Contract
  → candidatas a one-hot encoding. Varias incluyen un valor tipo "No internet service" en
  vez de un Sí/No puro — a revisar en el chequeo de consistencia interna.
- **Payment Method (4 valores)** → one-hot encoding.
- **Alta cardinalidad**: City (1,129), Lat Long (1,652) → confirmado, se descartan del modelo.
- **Churn Reason (20 valores)**: no se usa como predictora (data leakage temporal), pero es
  candidata para un gráfico exploratorio de las razones de fuga más frecuentes.

## 7. Consistencia interna

Hasta acá auditamos cada columna por separado (tipos, nulos, cardinalidad). En este paso
chequeamos si existen relaciones lógicas **entre columnas** que deberían cumplirse siempre.

La sospecha puntual: varias columnas de servicios (Online Security, Online Backup, Tech
Support, etc.) tienen 3 valores en vez de un Sí/No puro — probablemente incluyen una opción
tipo "No internet service" para los clientes que no tienen `Internet Service` contratado.
Si esto se confirma, no es un error de datos: es información legítima que hay que tener en
cuenta al decidir el encoding en la Etapa 3 (por ejemplo, evaluar si "No" y "No internet
service" deberían tratarse como la misma categoría o no).

In [15]:
df['Online Security'].value_counts()

Online Security
No                     3498
Yes                    2019
No internet service    1526
Name: count, dtype: int64

In [16]:
pd.crosstab(df['Internet Service'], df['Online Security'])

Online Security,No,No internet service,Yes
Internet Service,,,
DSL,1241,0,1180
Fiber optic,2257,0,839
No,0,1526,0


### Resultado: consistencia confirmada

Se confirma la relación lógica esperada entre `Internet Service` y las columnas de
servicios dependientes (ej. `Online Security`): los 1,526 clientes sin `Internet Service`
caen exactamente en la categoría "No internet service" en las columnas dependientes, sin
excepciones. El dataset no presenta inconsistencias internas en esta relación.

**Decisión para la Etapa 3**: evaluar si "No" y "No internet service" se combinan en una
sola categoría binaria (simplificando el encoding) o se mantienen separadas (preservando
la distinción de que el cliente ni siquiera tiene el servicio base). Esto se decide al
llegar a esa etapa, con el impacto que tenga sobre el modelo.

In [17]:
columnas_dependientes = ['Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies']

for col in columnas_dependientes:
    print(f"--- {col} ---")
    print(df.query("`Internet Service` == 'No'")[col].value_counts())
    print()

--- Online Backup ---
Online Backup
No internet service    1526
Name: count, dtype: int64

--- Device Protection ---
Device Protection
No internet service    1526
Name: count, dtype: int64

--- Tech Support ---
Tech Support
No internet service    1526
Name: count, dtype: int64

--- Streaming TV ---
Streaming TV
No internet service    1526
Name: count, dtype: int64

--- Streaming Movies ---
Streaming Movies
No internet service    1526
Name: count, dtype: int64



### Confirmación completa: consistencia en columnas dependientes de Internet Service

Se verificó la misma relación lógica en las 5 columnas restantes de servicios dependientes
(`Online Backup`, `Device Protection`, `Tech Support`, `Streaming TV`, `Streaming Movies`):
los 1,526 clientes sin `Internet Service` caen en el 100% de los casos en "No internet
service" en cada una de estas columnas, sin ninguna excepción. El dataset presenta
consistencia interna total en este bloque de variables.

## Cierre de la Etapa 2

El diccionario de datos presentado al inicio de este notebook es el resultado directo de
la auditoría de 7 pasos documentada arriba (auditoría de columnas, tipos de datos, nulos,
balance de clases, cardinalidad y consistencia interna). Ninguna decisión sobre qué columnas
usar o descartar fue asumida de antemano — todas están respaldadas por el análisis realizado.

El dataset queda listo para pasar a la Etapa 3 (Preparación de datos), donde se van a
ejecutar las transformaciones identificadas: conversión de `Total Charges` a numérico,
encoding de variables categóricas, y evaluación de la fusión de categorías "No" / "No
internet service".

# Visualización de hallazgos clave

Los 7 pasos anteriores auditaron la calidad y estructura del dataset. Esta sección
retoma tres relaciones ya identificadas en la exploración previa de Excel (Contract,
Payment Method y antigüedad vs. Churn) y las reproduce con gráficos generados por
código — a diferencia de las tablas dinámicas, estos gráficos son reproducibles,
reutilizables en el Streamlit final (Etapa 6), y quedan documentados como parte
permanente del análisis en vez de vivir solo en un archivo de Excel aparte.

Se trabaja sobre `df` (no `df_clean` de la Etapa 3), ya que acá las categorías
todavía están en su formato de texto original — más legible para los gráficos.

## Gráfico 1: Churn por tipo de contrato
**Qué vamos a hacer**: calcular la tasa de churn (%) para cada tipo de contrato, y mostrarla en un gráfico de barras.

Por qué es la vista más contundente que viste en Excel (42,7% vs 11,3% vs 2,8%) — reproducirla en código la deja lista para el README y el Streamlit final, y consolida con evidencia visual reproducible el hallazgo más fuerte del proyecto.

### Paso 1: calculamos la tasa de churn por grupo

In [18]:
churn_por_contrato = df.groupby('Contract')['Churn Value'].mean().reset_index()
churn_por_contrato['Churn Value'] = churn_por_contrato['Churn Value'] * 100
churn_por_contrato

,Contract,Churn Value
0,Month-to-month,42.709677
1,One year,11.269518
2,Two year,2.831858


#### Tabla de tasas de churn por Contract

Calculado con `groupby().mean()`, coincide exactamente con los valores observados en
la exploración de Excel (Month-to-month: 42.7%, One year: 11.3%, Two year: 2.8%).

### Paso 2: graficar la tabla con Plotly

In [19]:
import plotly.express as px

fig = px.bar(
    churn_por_contrato,
    x='Contract',
    y='Churn Value',
    title='Tasa de Churn por Tipo de Contrato',
    labels={'Churn Value': 'Tasa de Churn (%)', 'Contract': 'Tipo de Contrato'},
    text_auto='.1f'
)
fig.show()

### Resultado: gráfico de Churn por Contract

Se confirma visualmente la relación fuertemente inversa entre compromiso contractual y
fuga de clientes, ya identificada en Excel: Month-to-month (42.7%) muestra una tasa de
churn ~15 veces mayor que Two year (2.8%).

## Gráfico 2: Churn por método de pago

### Paso 1: Calculamos la tasa de churn por Método de Pago
Calculamos la tasa de churn (%) para cada método de pago, usando `groupby().mean()`,
el mismo enfoque aplicado en el gráfico anterior de Contract.

In [21]:
churn_por_pago = df.groupby('Payment Method')['Churn Value'].mean().reset_index()
churn_por_pago['Churn Value'] = churn_por_pago['Churn Value'] * 100
churn_por_pago

,Payment Method,Churn Value
0,Bank transfer (automatic),16.709845
1,Credit card (automatic),15.243101
2,Electronic check,45.285412
3,Mailed check,19.106700


### Paso 2: graficar la tabla con Plotly

In [23]:
fig = px.bar(
    churn_por_pago,
    x='Payment Method',
    y='Churn Value',
    title='Tasa de Churn por Tipo de Método de Pago',
    labels={'Churn Value': 'Tasa de Churn (%)', 'Payment Method': 'Método de Pago'},
    text_auto='.1f'
)
fig.show()

### Resultado: gráfico de Churn por Payment Method

Se confirma visualmente que Electronic check tiene una tasa de churn notablemente más
alta (45.3%) que los métodos de pago automáticos (Bank transfer 16.7%, Credit card
15.2%), sugiriendo que la fricción operativa del pago manual podría ser un factor de
fuga distinto al compromiso contractual.

## Gráfico 3: Churn por grupo de antigüedad

Este tiene una diferencia respecto a los dos anteriores: `Tenure Months` es numérica, no categórica — así que antes de agrupar necesitamos crear los rangos con `pd.cut`(), la misma función que ya usaste en el paso 7 de la Etapa 3 (para `Grupo_Antiguedad` en `df_clea`n). Ahora la vamos a recrear acá, pero sobre `df` (el original, sin tocar).

### Paso 1: crear los rangos de antigüedad sobre `df`

In [24]:
df['Grupo_Antiguedad'] = pd.cut(
    df['Tenure Months'],
    bins=[-1, 11, 23, 35, 47, 59, 72],
    labels=['0-11', '12-23', '24-35', '36-47', '48-59', '60-72']
)

In [25]:
df['Grupo_Antiguedad'].value_counts()

Grupo_Antiguedad
0-11     2069
60-72    1483
12-23    1047
24-35     876
48-59     820
36-47     748
Name: count, dtype: int64

### Resultado: Grupo_Antiguedad en df

Se recreó la misma categorización de antigüedad usada en la Etapa 3 (`pd.cut()` con los
mismos 6 rangos), esta vez sobre `df` para preservar las etiquetas de texto legibles.
Los conteos coinciden exactamente con los obtenidos previamente en `df_clean`.

In [26]:
churn_por_antiguedad = df.groupby('Grupo_Antiguedad')['Churn Value'].mean().reset_index()
churn_por_antiguedad['Churn Value'] = churn_por_antiguedad['Churn Value'] * 100 
churn_por_antiguedad

,Grupo_Antiguedad,Churn Value
0,0-11,48.284195
1,12-23,29.512894
2,24-35,22.031963
3,36-47,19.518717
4,48-59,15.000000
5,60-72,6.675657


### Tabla de tasas de churn por Grupo de Antigüedad

Calculado con `groupby().mean()`, confirma la relación monótona decreciente ya observada
en Excel: 48.3% de churn en el primer año (0-11 meses) vs. 6.7% en clientes de 5+ años
(60-72 meses).

### Paso 2: graficar la tabla con Plotly

In [29]:
fig = px.bar(
    churn_por_antiguedad,
    x='Grupo_Antiguedad',
    y='Churn Value',
    title='Tasa de Churn por Grupo de Antigüedad',
    labels={'Churn Value': 'Tasa de Churn (%)', 'Grupo_Antiguedad': 'Grupo de Antigüedad'},
    text_auto='.1f'
)
fig.show()

## Resultado: gráfico de Churn por Grupo de Antigüedad

Se confirma visualmente la relación monótona decreciente entre antigüedad y churn: el
riesgo de fuga cae de forma sostenida a medida que el cliente pasa más tiempo en la
empresa, concentrándose el mayor riesgo en el primer año (48.3%).

## Cierre: Visualización de hallazgos clave

Los tres gráficos reproducen y confirman con código reproducible los hallazgos ya
identificados en la exploración de Excel: Contract, Payment Method y antigüedad son
las tres variables con relación más fuerte y clara con el churn, cada una capturando
una causa de fuga distinta (compromiso contractual, fricción operativa, y riesgo del
período inicial del cliente). Estos tres gráficos quedan disponibles para reutilizar
en el README y en el dashboard de Streamlit (Etapa 6).